# 🚀 50M Bengali GPT — ২-স্টেজ প্রোডাকশন ট্রেনিং (Free Colab T4)
### 📊 স্টেপ ক্যালকুলেশন (Step Calculation):
```
Tokens/Step  = batch(4) × grad_accum(4) × context(512) = 8,192 tokens
Corpus Lines ≈ ১২,৮০,০০০ লাইন (~৩.৪৫ কোটি Train Tokens)
Stage 1 (2 ep) ≈ ৮,৪০০ স্টেপ (~৪০ মিনিট)
SFT Data     ≈ ৩,৮০,০০০ লাইন
Stage 2 (3 ep) ≈ ৩,৪৫০ স্টেপ (~১৫ মিনিট)
```
**নতুন স্পেশালাইজড ডোমেন (১,০০,০০০ লাইন):**
🇨🇳 চায়না সোর্সিং (1688, Alibaba, Yiwu) • এয়ার কার্গো/সি ফ্রেইট • সিঅ্যান্ডএফ এজেন্ট • কাস্টমস ও এলসি • বড় বনাম ছোট ব্যবসায়ীদের কৌশল • ছোট ব্যবসায়ীদের গ্রুপ বাইং/কনসোলিডেশন • চকবাজার/ইসলামপুর পাইকারি বাজার • হট সেলিং চাইনিজ প্রোডাক্ট • ফেসবুক পেজ ও ওয়েবসাইট সেলস ফানেল • প্রফিট মার্জিন ও ল্যান্ডিং কস্ট ক্যালকুলেশন


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
STAGE1_DIR = os.path.join(DRIVE_BASE_DIR, 'stage1_pretrain')
STAGE2_DIR = os.path.join(DRIVE_BASE_DIR, 'stage2_sft')
os.makedirs(STAGE1_DIR, exist_ok=True)
os.makedirs(STAGE2_DIR, exist_ok=True)
print(f'✓ Drive রেডি: {DRIVE_BASE_DIR}')

In [ ]:
%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git
%cd /content/ss_100m/ss_50million
!pip install -q -r requirements.txt
print('✓ পরিবেশ প্রস্তুত!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 4: বিশাল ডেটাসেট তৈরি (~১২.৮ লাখ লাইন)
# Sources:
#   ১. বাংলা Wikipedia                     → ৫,০০,০০০
#   ২. বাংলা Newspaper                     → ২,০০,০০০
#   ৩. English Wikipedia                    → ১,০০,০০০
#   ৪. Alpaca-Orca ইনস্ট্রাকশন             →    ৮০,০০০
#   ৫. নিশ ডোমেন (১০টি ক্যাটাগরি)          → ১,০০,০০০
#   ৬. চায়না সোর্সিং ও ই-কমার্স বিজনেস    → ১,০০,০০০
#   ৭. পাটিগণিত + ঐকিক নিয়ম + শতকরা      → ১,০০,০০০
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, re, json, random
from datasets import load_dataset

os.makedirs('data', exist_ok=True)
corpus_path   = 'data/corpus.txt'
sft_data_path = 'data/sft_data.txt'

need_download = True
if os.path.exists(corpus_path) and os.path.exists(sft_data_path):
    with open(corpus_path, encoding='utf-8') as f: c_cnt = sum(1 for _ in f)
    with open(sft_data_path, encoding='utf-8') as f: s_cnt = sum(1 for _ in f)
    if c_cnt >= 1150000 and s_cnt >= 250000:
        print(f'✓ সম্পূর্ণ ডেটাসেট বিদ্যমান (Corpus:{c_cnt:,} | SFT:{s_cnt:,}) — স্কিপ।')
        need_download = False
    else:
        print(f'⚠️ ডেটা অসম্পূর্ণ (Corpus:{c_cnt:,}, SFT:{s_cnt:,}) — রিডাউনলোড হচ্ছে...')

if need_download:
    corpus_lines, sft_lines = [], []
    print('='*70)
    print('📥 ১২.৮ লাখ লাইনের মেগা ডেটা সংগ্রহ শুরু হচ্ছে...')
    print('='*70)

    # ━━━ SOURCE ১: বাংলা Wikipedia (৫,০০,০০০) ━━━━━━━━━━━━━━━━━━━━━━━
    print('  [১/৭] বাংলা Wikipedia → ৫,০০,০০০...')
    wiki_bn = load_dataset('wikimedia/wikipedia','20231101.bn',split='train',streaming=True)
    bn_cnt  = 0
    for item in wiki_bn:
        for p in item.get('text','').split('\n'):
            p = p.strip()
            if len(p)>=25 and re.search(r'[\u0980-\u09FF]',p):
                corpus_lines.append(p); bn_cnt+=1
                if bn_cnt>=500000: break
        if bn_cnt>=500000: break
    print(f'     ✓ বাংলা উইকিপিডিয়া: {bn_cnt:,} লাইন')

    # ━━━ SOURCE ২: বাংলা News (২,০০,০০০) ━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print('  [২/৭] বাংলা Newspaper → ২,০০,০০০...')
    try:
        news_ds  = load_dataset('zabir-nabil/bangla_newspaper_dataset',split='train',streaming=True)
        news_cnt = 0
        for row in news_ds:
            txt = (row.get('text','') or row.get('content','')).strip()
            for p in txt.split('\n'):
                p = p.strip()
                if len(p)>=25 and re.search(r'[\u0980-\u09FF]',p):
                    corpus_lines.append(p); news_cnt+=1
                    if news_cnt>=200000: break
            if news_cnt>=200000: break
        print(f'     ✓ বাংলা সংবাদপত্র: {news_cnt:,} লাইন')
    except Exception as e:
        print(f'     ⚠️ News ফলব্যাক: {e}')

    # ━━━ SOURCE ৩: English Wikipedia (১,০০,০০০) ━━━━━━━━━━━━━━━━━━━━━
    print('  [৩/৭] English Wikipedia → ১,০০,০০০...')
    wiki_en = load_dataset('wikimedia/wikipedia','20231101.en',split='train',streaming=True)
    en_cnt  = 0
    for item in wiki_en:
        for p in item.get('text','').split('\n'):
            p = p.strip()
            if len(p)>=35 and re.search(r'[a-zA-Z]',p):
                corpus_lines.append(p); en_cnt+=1
                if en_cnt>=100000: break
        if en_cnt>=100000: break
    print(f'     ✓ English উইকিপিডিয়া: {en_cnt:,} লাইন')

    # ━━━ SOURCE ৪: Alpaca-Orca (৮০,০০০) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print('  [৪/৭] Bangla Alpaca-Orca → ৮০,০০০...')
    try:
        alpaca_ds = load_dataset('BanglaLLM/bangla-alpaca-orca',split='train',streaming=True)
        alp_cnt   = 0
        for row in alpaca_ds:
            inst = row.get('instruction','').strip()
            inp  = row.get('input','').strip()
            out  = row.get('output','').strip()
            if inst and out:
                fq = f'{inst} {inp}'.strip()
                sft_lines.append(f'প্রশ্ন: {fq} উত্তর: {out} <EOS>')
                corpus_lines.append(f'{fq} {out}')
                alp_cnt+=1
                if alp_cnt>=80000: break
        print(f'     ✓ Alpaca-Orca: {alp_cnt:,} জোড়া')
    except Exception as e:
        print(f'     ⚠️ Alpaca ফলব্যাক: {e}')

    # ━━━ SOURCE ৫: নিশ ডোমেন — ১০ ক্যাটাগরি (১,০০,০০০) ━━━━━━━━━━━━━━
    print('  [৫/৭] নিশ ডোমেন (১০ ক্যাটাগরি) → ১,০০,০০০...')
    def _add_niche(qa_pairs, n_each):
        for q, a in qa_pairs:
            for _ in range(n_each):
                corpus_lines.append(f'{q} {a}')
                sft_lines.append(f'প্রশ্ন: {q} উত্তর: {a} <EOS>')

    # 10 Niche Categories
    dm_qa = [('ডিজিটাল মার্কেটিং কী?','ডিজিটাল মাধ্যম ব্যবহার করে পণ্য বা সেবার প্রচার ও বিক্রি করার প্রক্রিয়া।'),('SEO কী?','ওয়েবসাইটকে সার্চ ইঞ্জিনের শীর্ষে দেখানোর কৌশল।'),('Facebook Ads কিভাবে কাজ করে?','টার্গেট অডিয়েন্স ও বাজেট নির্ধারণ করে ফেসবুক ও ইনস্টাগ্রামে বিজ্ঞাপন দেখানো হয়।')]*7
    nctb_qa = [('সালোকসংশ্লেষণ কী?','সূর্যালোক, পানি ও কার্বন ডাই-অক্সাইড ব্যবহার করে উদ্ভিদের গ্লুকোজ তৈরির প্রক্রিয়া।'),('মুক্তিযুদ্ধ কত সালে হয়েছিল?','১৯৭১ সালে। ২৬ মার্চ স্বাধীনতা ও ১৬ ডিসেম্বর বিজয় দিবস।')]*10
    health_qa = [('ডায়াবেটিস কী?','রক্তে স্বাভাবিকের চেয়ে বেশি শর্করার পরিমাণ থাকার রোগ।'),('উচ্চ রক্তচাপ নিয়ন্ত্রণের উপায়?','লবণ কম খাওয়া, নিয়মিত হাঁটা ও ওজন নিয়ন্ত্রণে রাখা।')]*10
    biz_qa = [('ব্যবসায়িক পরিকল্পনা কী?','ব্যবসার লক্ষ্য, পণ্য, মার্কেট ও আর্থিক পূর্বাভাসের সুনির্দিষ্ট লিখিত রূপ।'),('ক্যাশ ফ্লো কী?','ব্যবসায়ে অর্থের আগমন ও বহির্গমনের নিখুঁত হিসাব।')]*10
    tech_qa = [('Python কেন শিখব?','সহজ সিনট্যাক্স, ওয়েব ডেভেলপমেন্ট ও এআই-এর জন্য বিশ্বজুড়ে সবচেয়ে জনপ্রিয়।'),('AI কী?','মানুষের বুদ্ধিমত্তার মতো চিন্তা ও সিদ্ধান্ত নেওয়ার কম্পিউটার সক্ষমতা।')]*10
    agri_qa = [('কম্পোস্ট সার কী?','জৈব বর্জ্য ও গোবর পচিয়ে তৈরি পরিবেশবান্ধব প্রাকৃতিক সার।'),('ধান চাষের সেরা সময়?','বোরো নভেম্বরের শেষে এবং আমন বর্ষাকালে রোপণ করা হয়।')]*10
    islamic_qa = [('নামাজ কয় ওয়াক্ত?','পাঁচ ওয়াক্ত: ফজর, জোহর, আসর, মাগরিব ও এশা।'),('যাকাতের বিধান কী?','নির্দিষ্ট নিসাব পরিমাণ সম্পদের ২.৫% দরিদ্রদের দান করা ফরজ।')]*10
    recipe_qa = [('বিরিয়ানি মশলায় কী কী থাকে?','এলাচ, দারচিনি, লবঙ্গ, জায়ফল, জয়িত্রী, তেজপাতা ও স্টার অ্যানিস।'),('ভাপা পিঠা কীভাবে তৈরি হয়?','চালের গুঁড়া, গুড় ও নারিকেল দিয়ে ভাপে সিদ্ধ করে।')]*10
    law_qa = [('ট্রেড লাইসেন্স কী?','সিটি কর্পোরেশন বা পৌরসভা থেকে প্রাপ্ত ব্যবসা পরিচালনার বৈধ সনদ।'),('ভোক্তা অধিকার হটলাইন কত?','ভোক্তা অধিকার সংরক্ষণ অধিদপ্তরের হটলাইন নম্বর ১৬১২১।')]*10
    gk_qa = [('পদ্মা সেতুর দৈর্ঘ্য কত?','পদ্মা সেতুর মূল দৈর্ঘ্য ৬.১৫ কিলোমিটার।'),('বাংলাদেশের জাতীয় সংগীতের রচয়িতা কে?','কবিগুরু রবীন্দ্রনাথ ঠাকুর।')]*10

    for cat_qa in [dm_qa, nctb_qa, health_qa, biz_qa, tech_qa, agri_qa, islamic_qa, recipe_qa, law_qa, gk_qa]:
        _add_niche(cat_qa[:20], 500)
    print('     ✓ ১০টি নিশ ডোমেন: ১,০০,০০০ লাইন সম্পন্ন')

    # ━━━ SOURCE ৬: চায়না সোর্সিং ও ই-কমার্স বিজনেস (১,০০,০০০ লাইন) ━━━
    print('  [৬/৭] চায়না সোর্সিং ও বাংলাদেশ ই-কমার্স → ১,০০,০০০ লাইন...')
    china_core_qa = [
        ('চায়না থেকে বাংলাদেশে পণ্য সোর্স করার প্রধান প্ল্যাটফর্ম কোনগুলো?',
         'চায়না থেকে পণ্য কেনার প্রধান ওয়েবসাইটগুলো হলো 1688.com (স্থানীয় পাইকারি দর, সবচেয়ে সস্তা), Alibaba.com (আন্তর্জাতিক বায়ারদের জন্য ট্রেড অ্যাসুরেন্সযুক্ত), Taobao.com (খুচরা ও ছোট স্যাম্পল) এবং Made-in-China.com (ভারী যন্ত্রপাতি ও ফ্যাক্টরি ডিরেক্টরি)।'),
        ('1688.com থেকে কীভাবে পণ্য কিনতে হয়?',
         '1688 মূলত চাইনিজ ভাষার ডোমেস্টিক সাইট। গুগল ক্রোম দিয়ে অনুবাদ করে প্রোডাক্ট রিসার্চ করতে হয়। পেমেন্টের জন্য চাইনিজ ব্যাংক অ্যাকাউন্ট বা支付宝 (Alipay) লাগে। বাংলাদেশের শিপিং বা বায়িং এজেন্টদের লিংক দিলে তারা পণ্য কিনে নিজেদের ওয়্যারহাউসে রিসিভ করে বাংলাদেশে পাঠিয়ে দেয়।'),
        ('বড় ব্যবসায়ীরা চায়না থেকে কীভাবে পণ্য আমদানি করে?',
         'বড় ব্যবসায়ীরা সরাসরি ফ্যাক্টরি পরিদর্শন করে বা ক্যান্টন ফেয়ারে গিয়ে চুক্তি করে। এরপর ব্যাংকে এলসি (Letter of Credit) খুলে ফুল কন্টেইনার (FCL - 20ft বা 40ft) সি-ফ্রেইটে জাহাজে পাঠায়। চট্টগ্রাম বা মংলা বন্দরে সিঅ্যান্ডএফ (C&F) এজেন্ট নিয়োগ করে শুল্ক-কর (Duty & Taxes) পরিশোধের পর পণ্য ওয়্যারহাউসে নেয়।'),
        ('ছোট ও নতুন ব্যবসায়ীরা চায়না থেকে কীভাবে পণ্য আনে?',
         'ছোট ব্যবসায়ীরা এলসি না খুলে সিঅ্যান্ডএফ শিপিং এজেন্টের মাধ্যমে ডোর-টু-ডোর (Door-to-Door D2D) সার্ভিসে পণ্য আনে। শিপিং এজেন্টরা কেজি বা সিবিএম (CBM) হিসেবে রেট নেয় এবং সব কাস্টমস ও ট্যাক্স ক্লিয়ার করে সরাসরি ঢাকায় পণ্য পৌঁছে দেয়।'),
        ('এয়ার কার্গো বনাম সি ফ্রেইট (বাই এয়ার বনাম বাই সি)-এর পার্থক্য কী?',
         'বাই এয়ার (Air Cargo): ৭ থেকে ১০ দিনে আসে, খরচ প্রতি কেজি ৮০০ থেকে ১,২০০ টাকা (হালকা, ট্রেন্ডি ও হাই-ভ্যালু পণ্যের জন্য উপযোগী)। বাই সি (Sea Freight LCL): ৩৫ থেকে ৫০ দিন সময় লাগে, খরচ প্রতি কেজি ২০০ থেকে ৩৫০ টাকা বা প্রতি CBM এ ১৬,০০০ থেকে ২৪,০০০ টাকা (ভারী ও কম দামের পণ্যের জন্য উপযোগী)।'),
        ('ছোট ব্যবসায়ীরা কীভাবে একত্রিত হয়ে চায়না থেকে পণ্য আনতে পারে (Group Buying)?',
         'ছোট উদ্যোক্তারা ফেসবুক গ্রুপ বা কমিউনিটিতে যুক্ত হয়ে গ্রুপ বাইং বা কনসোলিডেশন করে। ৫-১০ জন মিলে একটি কার্টুন বা সম্পূর্ণ প্যালেট অর্ডার করে। এতে ফ্যাক্টরি থেকে সর্বোচ্চ ডিসকাউন্ট পাওয়া যায় এবং শিপিং চার্জ কেজিপ্রতি অনেক কমে আসে।'),
        ('চায়না থেকে না এনে ঢাকার স্থানীয় পাইকারি বাজার থেকে কীভাবে চাইনিজ পণ্য সোর্স করা যায়?',
         'নতুন ব্যবসায়ীদের জন্য চকবাজার (প্লাস্টিক, গিফট, খেলনা, কসমেটিক্স), ইসলামপুর (পোশাক ও ফেব্রিক্স), নবাবপুর (ইলেকট্রনিক্স ও হার্ডওয়্যার), এবং স্টেডিয়াম মার্কেট/মোতালিব প্লাজা (মোবাইল এক্সেসরিজ) হলো প্রধান চাইনিজ পাইকারি হাব। এখানে যাচাই করে নগদে স্বল্প পুঁজিতে পণ্য নেওয়া যায়।'),
        ('অনলাইনে বা ফেসবুক পেজে বিক্রির জন্য সবচেয়ে সেরা চাইনিজ পণ্য কোনগুলো?',
         '১. স্মার্ট গ্যাজেট: TWS ইয়ারবাডস, স্মার্টওয়াচ, রিং লাইট, পোর্টেবল ব্লেন্ডার, মিনি ফ্যান। ২. কিচেন অ্যাপ্লায়েন্স: ইলেকট্রিক চপার, এগ বয়লার, সিলিং মেশিন। ৩. বিউটি ও স্কিনকেয়ার টুলস: ব্ল্যাকহেড রিমুভার, মেকআপ ব্রাশ, ফেসিয়াল ম্যাসাজার। ৪. ইউনিক খেলনা ও কিডস আইটেম। ৫. ট্রেন্ডি জুয়েলারি ও ফ্যাশন এক্সেসরিজ।'),
        ('ফেসবুক পেজ দিয়ে চাইনিজ পণ্য সফলভাবে বিক্রির কার্যকর কৌশল কী?',
         'পণ্য ব্যবহারের বাস্তব সমস্যা সমাধানের ভিডিও বা রিলস বানান। ফেসবুক বিজ্ঞাপন (Engagement + Sales Campaign) চালান যাতে মেসেজ ও অর্ডারের কস্ট কম থাকে। একটি প্রফেশনাল ল্যান্ডিং পেজে ১-ক্লিক ক্যাশ অন ডেলিভারি ফর্ম রাখুন এবং স্টিডফাস্ট বা পাঠাও কুরিয়ারের মাধ্যমে দ্রুত ডেলিভারি নিশ্চিত করুন।'),
        ('চাইনিজ পণ্যের ল্যান্ডিং কস্ট (Landing Cost) এবং প্রফিট মার্জিন কীভাবে হিসাব করবেন?',
         'ল্যান্ডিং কস্ট = (চায়না পণ্যের ক্রয়মূল্য + চায়না লোকাল কুরিয়ার + এয়ার/সি শিপিং চার্জ প্রতি কেজি + কাস্টমস ক্লিয়ারিং)। এর সাথে বিজ্ঞাপন খরচ (প্রতি অর্ডারে ৮০-১২০ টাকা) ও রিটার্ন লস যোগ করে বিক্রয়মূল্য ঠিক করতে হয়। সাধারণত চাইনিজ গ্যাজেটে ৫০% থেকে ১৫০% পর্যন্ত গ্রস প্রফিট মার্জিন রাখা যায়।'),
        ('সাপ্লায়ার বাছাইয়ে কী কী সতর্কতা অবলম্বন করতে হয়?',
         'সাপ্লায়ারের ট্রেড অ্যাসুরেন্স আছে কিনা, অন্তত ৩ বছরের পুরনো ভেরিফায়েড প্রোফাইল কিনা এবং কাস্টমার রিভিউ স্কোর ৪.৮+ কিনা তা দেখা জরুরি। বড় অর্ডারের আগে অবশ্যই ১-২ পিস পেইড স্যাম্পল এনে কোয়ালিটি টেস্ট করতে হবে।'),
        ('অনলাইনে ক্যাশ অন ডেলিভারি (COD) ব্যবসায় রিটার্ন রেশিও কীভাবে কমাবেন?',
         'অর্ডার আসার সাথে সাথে ফোনে কথা বলে অ্যাড্রেস কনফার্ম করুন। ডেলিভারি চার্জের কিছু অংশ অগ্রিম নিলে ফেক অর্ডার ৯০% কমে যায়। কুরিয়ারে পণ্য ট্র্যাকিং করে কাস্টমারকে এসএমএস দিয়ে আপডেট জানান।'),
        (' ড্রপশিপিং কীভাবে কাজ করে চায়না পণ্যের ক্ষেত্রে?',
         'ড্রপশিপিংয়ে বিক্রেতাকে কোনো পণ্য আগে থেকে কিনে স্টকে রাখতে হয় না। কাস্টমার থেকে অর্ডার পাওয়ার পর সরাসরি চায়না বা লোকাল হোলসেলারকে অর্ডার ও কাস্টমারের ঠিকানা দেওয়া হয়, তারা সরাসরি কাস্টমারের কাছে পাঠিয়ে দেয়।'),
        ('ই-কমার্সে ওয়েবসাইট বা ল্যান্ডিং পেজ থাকা কেন জরুরি?',
         'ওয়েবসাইট থাকলে কাস্টমার সরাসরি পণ্য দেখে অর্ডার দিতে পারে, ইনবক্সে ঘণ্টার পর ঘণ্টা চ্যাট করতে হয় না। ফেসবুক পিক্সেল ও কনভার্সন এপিআই সঠিকভাবে ডেটা ট্র্যাক করতে পারে, ফলে বিজ্ঞাপনের রিটার্ন (ROAS) অনেক বেশি আসে।'),
        ('ক্যান্টন ফেয়ার (Canton Fair) কী এবং এতে কীভাবে অংশ নেওয়া যায়?',
         'ক্যান্টন ফেয়ার হলো বিশ্বের বৃহত্তম ট্রেড ফেয়ার যা প্রতি বছর এপ্রিলে ও অক্টোবরে চীনের গুয়াংজুতে অনুষ্ঠিত হয়। এখানে বিশ্বের লাখ লাখ ফ্যাক্টরি অংশ নেয়। পাসপোর্ট, ভিসা ও বায়ার ব্যাজ নিয়ে গিয়ে সরাসরি ফ্যাক্টরি মালিকদের সাথে আলোচনা করা যায়।')
    ]

    # পণ্যের ক্যাটাগরি ও প্যারামিটারাইজড ভ্যারিয়েশন তৈরি
    products_list = [
        ('স্মার্টওয়াচ', 'গ্যাজেট', '৮৫০', '১,৪৫০', 'এয়ার কার্গো', 'ফেসবুক ভিডিও বিজ্ঞাপন ও রিলস'),
        ('TWS ব্লুটুথ এয়ারবাডস', 'গ্যাজেট', '৪২০', '৮৫০', 'এয়ার কার্গো', 'অফার কম্বো ও স্টুডেন্ট ডিসকাউন্ট'),
        ('রিচার্জেবল মিনি জুসার ব্লেন্ডার', 'কিচেন গ্যাজেট', '৫৫০', '১,১৯০', 'সি ফ্রেইট LCL', 'রান্না ও ফিটনেস পেজে প্রমোশন'),
        ('হ্যান্ডহেল্ড ইলেকট্রিক চপার', 'কিচেন গ্যাজেট', '৩২০', '৬৫০', 'সি ফ্রেইট LCL', 'রান্নার ভিডিও ও গৃহিণীদের টার্গেট অ্যাড'),
        ('পোর্টেবল নেক ফ্যান', 'সিজনাল আইটেম', '৩৮০', '৭৫০', 'এয়ার কার্গো', 'গ্রীষ্মকালে ফেসবুক বুস্টিং'),
        ('সেলফি রিং লাইট ও ট্রাইপড', 'কনটেন্ট ক্রিয়েশন', '৫০০', '১,০৫০', 'সি ফ্রেইট LCL', 'টিকটকার ও ইউটিউবারদের টার্গেট বিজ্ঞাপন'),
        ('ব্ল্যাকহেড রিমুভার ভিশন মেশিন', 'বিউটি কেয়ার', '৪৫০', '১,১০০', 'এয়ার কার্গো', 'বিউটি ব্লগারদের রিভিউ ও ইনস্টাগ্রাম রিলস'),
        ('কোরিয়ান স্টাইল ট্রেন্ডি জুয়েলারি সেট', 'ফ্যাশন', '১৫০', '৪৫০', 'এয়ার কার্গো', 'ফেসবুক লাইভ ও ক্যাটালগ অ্যাড'),
        ('ম্যাজিক ওয়াটার ড্রয়িং বুক', 'কিডস আইটেম', '১২০', '৩২০', 'সি ফ্রেইট LCL', 'অভিভাবকদের গ্রুপ ও ফেসবুক পেজে প্রচার'),
        ('পোর্টেবল কার ভ্যাকুয়াম ক্লিনার', 'অটোমোবাইল', '৬৫০', '১,৩৫০', 'সি ফ্রেইট LCL', 'গাড়ির মালিকদের ফেসবুক অডিয়েন্স টার্গেটিং')
    ]

    templates_sourcing = [
        ('{prod} চায়না থেকে কীভাবে সোর্স করব এবং অনলাইনে কীভাবে বিক্রি করব?',
         '{prod} ({cat}) সোর্স করার জন্য 1688 বা আলিবাবায় ভেরিফায়েড সাপ্লায়ার খুঁজুন। প্রাথমিক খরচ আনুমানিক {buy} টাকা। {ship}-এ বাংলাদেশে আনতে খরচ হবে। ফেসবুকে {mkt}-এর মাধ্যমে এটি {sell} টাকায় বিক্রি করে ভালো প্রফিট করা সম্ভব।'),
        ('{prod} ব্যবসা শুরু করতে ন্যূনতম কত মূলধন দরকার এবং বড় বনাম ছোট ব্যবসায়ীদের কৌশল কী?',
         '{prod}-এর ছোট লট (২০-৫০ পিস) দিয়ে শুরু করতে ১৫,০০০ থেকে ২৫,০০০ টাকা মূলধন যথেষ্ট। নতুনরা এয়ার কার্গো বা ঢাকার চকবাজার পাইকারি বাজার থেকে আনবে, আর বড় ব্যবসায়ীরা কন্টেইনারে এনে প্রতি পিসে আরও ৩০-৪০% খরচ বাঁচায়।'),
        ('ছোট ব্যবসায়ীদের সাথে কনসোলিডেশন বা গ্রুপ বাইং করে {prod} কীভাবে আনা যায়?',
         'ফেসবুকে ই-কমার্স কমিউনিটির ৫-১০ জন উদ্যোক্তা মিলে {prod}-এর ৫০০-১০০০ পিসের বাল্ক অর্ডার একত্রিত করুন। এতে ফ্যাক্টরি থেকে পাইকারি দর {buy} টাকার নিচে নেমে আসবে এবং শিপিং এজেন্টের মাধ্যমে এক চালানে কম খরচে আনা যাবে।'),
        ('{prod} বিক্রির ক্ষেত্রে ওয়েবসাইট ও ডেলিভারি কীভাবে ম্যানেজ করব?',
         '{prod}-এর জন্য একটি ক্লিন ল্যান্ডিং পেজ বানান। অর্ডার আসার পর কাস্টমারকে ফোন করে ঠিকানা যাচাই করুন। স্টিডফাস্ট বা পাঠাও কুরিয়ারের মাধ্যমে ক্যাশ অন ডেলিভারিতে দ্রুত পৌঁছে দিন যাতে রিটার্ন রেশিও ৫% এর নিচে থাকে।')
    ]

    china_cnt = 0
    # প্রথমে কোর কিউএ যোগ করা
    for q, a in china_core_qa:
        for _ in range(1200):
            corpus_lines.append(f'{q} {a}')
            sft_lines.append(f'প্রশ্ন: {q} উত্তর: {a} <EOS>')
            china_cnt += 1

    # টেমপ্লেট ভ্যারিয়েশন দিয়ে মোট ১,০০,০০০ পূরণ করা
    for tmpl_q, tmpl_a in templates_sourcing:
        for prod, cat, buy, sell, ship, mkt in products_list:
            q_filled = tmpl_q.replace('{prod}', prod).replace('{cat}', cat)
            a_filled = tmpl_a.replace('{prod}', prod).replace('{cat}', cat).replace('{buy}', buy).replace('{sell}', sell).replace('{ship}', ship).replace('{mkt}', mkt)
            repeats = int((100000 - china_cnt) / (len(templates_sourcing) * len(products_list))) + 1
            for _ in range(repeats):
                if china_cnt >= 100000: break
                corpus_lines.append(f'{q_filled} {a_filled}')
                sft_lines.append(f'প্রশ্ন: {q_filled} উত্তর: {a_filled} <EOS>')
                china_cnt += 1
            if china_cnt >= 100000: break
        if china_cnt >= 100000: break
    print(f'     ✓ চায়না সোর্সিং ও ই-কমার্স: {china_cnt:,} লাইন')

    # ━━━ SOURCE ৭: পাটিগণিত ও ঐকিক নিয়ম (১,০০,০০০) ━━━━━━━━━━━━━━━━━
    print('  [৭/৭] পাটিগণিত + জেনারেল গণিত → ১,০০,০০০...')
    random.seed(42)
    math_types = ['add','sub','mul','div','percent','unitary','profit','en_math']
    for _ in range(100000):
        m = random.choice(math_types)
        if m == 'add':
            a,b = random.randint(5,9999),random.randint(5,9999)
            q,ans = f'{a} এর সাথে {b} যোগ করলে কত হয়?', f'{a} + {b} = {a+b}।'
        elif m == 'sub':
            a,b = random.randint(50,9999),random.randint(5,4999)
            if a<b: a,b=b,a
            q,ans = f'{a} থেকে {b} বিয়োগ করলে কত থাকে?', f'{a} - {b} = {a-b}।'
        elif m == 'mul':
            a,b = random.randint(2,999),random.randint(2,99)
            q,ans = f'{a} কে {b} দিয়ে গুণ করলে কত?', f'{a} × {b} = {a*b}।'
        elif m == 'div':
            d = random.randint(2,30); qv = d*random.randint(2,100)
            q,ans = f'{qv} কে {d} দিয়ে ভাগ করলে কত?', f'{qv} ÷ {d} = {qv//d}।'
        elif m == 'percent':
            base = random.choice([50,100,200,500,1000,2000,5000])
            rate = random.choice([5,10,15,20,25,30,50])
            val  = int(base*rate/100)
            q,ans = f'{base} টাকার {rate}% কত?', f'{base} × {rate}/100 = {val} টাকা।'
        elif m == 'unitary':
            n1,up = random.randint(2,8),random.randint(5,50)
            c1,n2 = n1*up,random.randint(9,20); c2=n2*up
            q,ans = (f'{n1}টি জিনিসের দাম {c1} টাকা হলে {n2}টির দাম কত?',
                     f'১টির দাম {c1}÷{n1}={up} টাকা। {n2}টির দাম {up}×{n2}={c2} টাকা।')
        elif m == 'profit':
            cp = random.randint(100,5000); prof=random.randint(10,cp//2); sp=cp+prof
            q,ans = (f'একটি জিনিস {cp} টাকায় কিনে {sp} টাকায় বিক্রি করলে লাভ কত?',
                     f'লাভ = {sp} - {cp} = {prof} টাকা।')
        else:
            a,b = random.randint(2,99),random.randint(2,50)
            q,ans = f'What is {a} multiplied by {b}?', f'{a} × {b} = {a*b}.'
        corpus_lines.append(f'{q} {ans}')
        sft_lines.append(f'প্রশ্ন: {q} উত্তর: {ans} <EOS>')
    print('     ✓ পাটিগণিত ও সাধারণ গণিত: ১,০০,০০০ লাইন')

    # ━━━ শাফেল ও ফাইলে লেখা ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print('⚡ ডেটা শাফেল ও ফাইলে লেখা হচ্ছে...')
    random.shuffle(corpus_lines)
    random.shuffle(sft_lines)
    with open(corpus_path,'w',encoding='utf-8') as f:
        for l in corpus_lines: f.write(l+'\n')
    with open(sft_data_path,'w',encoding='utf-8') as f:
        for l in sft_lines: f.write(l+'\n')
    for old in ['data/corpus_tokens.bin','data/sft_data_tokens.bin']:
        if os.path.exists(old): os.remove(old)
    c_mb = os.path.getsize(corpus_path)/(1024*1024)
    s_mb = os.path.getsize(sft_data_path)/(1024*1024)
    print('='*70)
    print(f'✅ CORPUS  : {len(corpus_lines):>10,} লাইন  ({c_mb:.1f} MB)')
    print(f'✅ SFT     : {len(sft_lines):>10,} লাইন  ({s_mb:.1f} MB)')
    print('='*70)

    # ━━━ লাইভ স্টেপ ক্যালকুলেশন ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    BATCH=4; GRAD=4; BLOCK=512; AVG_TOK=30; SPLIT=0.9; EP=2
    est_tok = len(corpus_lines)*AVG_TOK*SPLIT
    tps = BATCH*GRAD*BLOCK
    spe = int(est_tok/tps)
    total = spe*EP
    print(f'\n📊 স্টেজ-১ স্টেপ ক্যালকুলেশন:')
    print(f'   মোট কর্পাস লাইন  : {len(corpus_lines):,}')
    print(f'   আনুমানিক টোকেন   : {int(est_tok):,}')
    print(f'   Tokens/Step       : {tps:,}')
    print(f'   Steps/Epoch       : {spe:,}')
    print(f'   মোট স্টেপ (২ ep)  : {total:,}')
    print(f'   সময় (T4 ~3.5it/s): ~{total//3//60} মিনিট')

    # স্টেজ ২ হিসাব
    sft_spe = max(1, int((len(sft_lines)*AVG_TOK*SPLIT)/tps))
    sft_total = int(sft_spe * 3)
    print(f'\n🎯 স্টেজ-২ (SFT) হিসাব:')
    print(f'   মোট SFT লাইন     : {len(sft_lines):,}')
    print(f'   Steps/Epoch (SFT) : {sft_spe:,}')
    print(f'   মোট SFT স্টেপ(৩ ep): {sft_total:,}')
    print(f'   সময় (T4 ~3.5it/s): ~{sft_total//3//60} মিনিট')


In [ ]:
# Step 5: টোকেনাইজার তৈরি
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
special_tokens = ['<PAD>','<UNK>','<BOS>','<EOS>','<|system|>','<|user|>','<|assistant|>','<|math|>']
tok = Tokenizer(models.BPE(unk_token='<UNK>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=10000, special_tokens=special_tokens,
                               min_frequency=2, show_progress=True)
print('⚡ সম্পূর্ণ ডেটাসেটে টোকেনাইজার ট্রেনিং...')
tok.train(['data/corpus.txt','data/sft_data.txt'], trainer)
tok.save('tokenizer.json')
print(f'✓ Vocab Size: {tok.get_vocab_size():,}')

In [ ]:
# Step 6: 🔥 Stage 1 — Pretraining (2 epochs)
import os, glob, time, math, torch
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

tokenizer      = Tokenizer.from_file('tokenizer.json')
dataset_stage1 = BengaliDataset(corpus_path='data/corpus.txt', tokenizer=tokenizer,
                                 block_size=GPTConfig.block_size, split_ratio=0.9)
tps  = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
spe  = dataset_stage1.train_len // tps
EPOCHS = 2.0
max_iters = int(spe * EPOCHS)

print(f'📊 Stage-1 Live Info:')
print(f'   Train Tokens  : {dataset_stage1.train_len:,}')
print(f'   Tokens/Step   : {tps:,}')
print(f'   Steps/Epoch   : {spe:,}')
print(f'   Total (2 ep)  : {max_iters:,}')
print(f'   ~Time (T4)    : ~{max_iters//3//60} min')

raw_model = GPT(GPTConfig).to(device)
try: model = torch.compile(raw_model); print('✓ torch.compile সক্রিয়')
except: model = raw_model

optimizer = torch.optim.AdamW(raw_model.parameters(),
    lr=GPTConfig.learning_rate, betas=(0.9,0.95), weight_decay=0.1)
scaler = GradScaler()

start = 1
ckpts = glob.glob(os.path.join(STAGE1_DIR,'stage1_step_*.pt'))
if ckpts:
    def _s(p):
        try: return int(p.split('_step_')[-1].replace('.pt',''))
        except: return 0
    lc = sorted(ckpts, key=_s)[-1]; ls = _s(lc)
    if 0 < ls < max_iters:
        raw_model.load_state_dict(torch.load(lc, map_location=device))
        start = ls+1; print(f'🔄 Resume step {start:,}')

def get_lr(it, mx, lr=3e-4, mlr=3e-5):
    w=400
    if it<w: return lr*it/w
    if it>mx: return mlr
    return mlr+.5*(1+math.cos(math.pi*(it-w)/(mx-w)))*(lr-mlr)

print(f'🔥 Stage-1: {start:,} → {max_iters:,}')
model.train(); optimizer.zero_grad(set_to_none=True); t0=time.time()

for step in range(start, max_iters+1):
    lr = get_lr(step, max_iters, GPTConfig.learning_rate, GPTConfig.min_lr)
    for g in optimizer.param_groups: g['lr']=lr
    acc=0.0
    for _ in range(GPTConfig.gradient_accumulation_steps):
        x,y = dataset_stage1.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _,loss = model(x, targets=y); loss = loss/GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward(); acc+=loss.item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    if step%250==0 or step==start:
        ep=(step*tps)/dataset_stage1.train_len; ela=time.time()-t0
        spd=(step-start+1)/ela if ela>0 else 0; eta=(max_iters-step)/spd/60 if spd>0 else 0
        print(f'[S1] {step:5d}/{max_iters} Ep{ep:.2f} | Loss:{acc:.4f} | LR:{lr:.2e} | {spd:.2f}it/s ETA:{eta:.1f}m')
    if step%500==0 or step==max_iters:
        torch.save(raw_model.state_dict(), os.path.join(STAGE1_DIR,f'stage1_step_{step}.pt'))

final1=os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_1.pt')
torch.save(raw_model.state_dict(), final1)
print(f'🎉 Stage 1 সম্পন্ন! → {final1}')

In [ ]:
# Step 7: 🎯 Stage 2 — SFT (3 epochs)
import os, glob, time, math, torch
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
s1 = os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_1.pt')
raw_sft = GPT(GPTConfig).to(device)
if os.path.exists(s1):
    raw_sft.load_state_dict(torch.load(s1, map_location=device))
    print(f'✓ Stage-1 লোড: {s1}')
else: print('⚠️ Stage-1 নেই, scratch থেকে শুরু!')

try: sft_model=torch.compile(raw_sft); print('✓ torch.compile সক্রিয়')
except: sft_model=raw_sft

tokenizer      = Tokenizer.from_file('tokenizer.json')
dataset_stage2 = BengaliDataset(corpus_path='data/sft_data.txt', tokenizer=tokenizer,
                                 block_size=GPTConfig.block_size, split_ratio=0.9)
tps  = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
spe  = max(1, dataset_stage2.train_len // tps)
EPOCHS2=3.0; max_iters2=int(spe*EPOCHS2)

print(f'📊 Stage-2 SFT: Tokens={dataset_stage2.train_len:,} | Steps/Ep={spe:,} | Total={max_iters2:,}')

optimizer=torch.optim.AdamW(raw_sft.parameters(),
    lr=GPTConfig.sft_learning_rate, betas=(0.9,0.95), weight_decay=0.1)
scaler=GradScaler()

start2=1
sft_ckpts=glob.glob(os.path.join(STAGE2_DIR,'stage2_step_*.pt'))
if sft_ckpts:
    def _ss(p):
        try: return int(p.split('_step_')[-1].replace('.pt',''))
        except: return 0
    lc2=sorted(sft_ckpts,key=_ss)[-1]; ls2=_ss(lc2)
    if 0<ls2<max_iters2:
        raw_sft.load_state_dict(torch.load(lc2,map_location=device))
        start2=ls2+1; print(f'🔄 SFT Resume {start2:,}')

def get_sft_lr(it,mx,lr=1e-4,mlr=1e-5):
    w=min(100,mx//10)
    if it<w: return lr*it/w
    if it>mx: return mlr
    return mlr+.5*(1+math.cos(math.pi*(it-w)/max(1,mx-w)))*(lr-mlr)

print(f'🔥 Stage-2 SFT: {start2:,} → {max_iters2:,} স্টেপ')
sft_model.train(); optimizer.zero_grad(set_to_none=True); t0=time.time()

for step in range(start2, max_iters2+1):
    lr=get_sft_lr(step,max_iters2,GPTConfig.sft_learning_rate,GPTConfig.sft_min_lr)
    for g in optimizer.param_groups: g['lr']=lr
    acc=0.0
    for _ in range(GPTConfig.gradient_accumulation_steps):
        x,y=dataset_stage2.get_batch('train',batch_size=GPTConfig.batch_size,device=device)
        with autocast(dtype=torch.float16):
            _,loss=sft_model(x,targets=y); loss=loss/GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward(); acc+=loss.item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_sft.parameters(), 1.0)
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    if step%25==0 or step==start2 or step==max_iters2:
        ep=(step*tps)/dataset_stage2.train_len; ela=time.time()-t0
        spd=(step-start2+1)/ela if ela>0 else 0
        print(f'[S2] {step:4d}/{max_iters2} Ep{ep:.2f} | Loss:{acc:.4f} | LR:{lr:.2e} | {spd:.2f}it/s')
    if step%100==0 or step==max_iters2:
        torch.save(raw_sft.state_dict(), os.path.join(STAGE2_DIR,f'stage2_step_{step}.pt'))

final2=os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_2_final.pt')
torch.save(raw_sft.state_dict(), final2)
print(f'🎉 চূড়ান্ত মডেল → {final2}')

In [ ]:
# Step 8: 💬 চ্যাটবট টেস্ট (চায়না সোর্সিং ও বিজনেস সহ)
import torch
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
final2 = os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_2_final.pt')
chat_model = GPT(GPTConfig).to(device)
chat_model.load_state_dict(torch.load(final2, map_location=device))
chat_model.eval()
tokenizer = Tokenizer.from_file('tokenizer.json')

# ✏️ আপনার টেস্ট প্রশ্ন লিখুন:
user_question = 'চায়না থেকে বাংলাদেশে পণ্য সোর্স করার প্রধান প্ল্যাটফর্ম কোনগুলো এবং ছোট ব্যবসায়ীরা কীভাবে গ্রুপ বাইং করে?'

prompt = f'প্রশ্ন: {user_question} উত্তর:'
enc    = tokenizer.encode(prompt)
ids    = enc.ids if hasattr(enc,'ids') else enc
inp    = torch.tensor([ids], dtype=torch.long, device=device)
eos_id = tokenizer.token_to_id('<EOS>')

with torch.no_grad():
    out = chat_model.generate(inp, max_new_tokens=250,
                               temperature=0.7, top_k=40,
                               repetition_penalty=1.25, eos_id=eos_id)
reply = tokenizer.decode(out[0].cpu().tolist()).split('<EOS>')[0].strip()
print('='*60)
print(reply)
print('='*60)